# Programming Assignment 2: Text Representation - Classic Method
**Twitter Entity Sentiment - bag of words, TF-IDF and n-grams**

## 📋 Assignment Information

**Course:** Unstructured Data Analysis (2026-2)
**Assignment:** Programming Assignment 2 - Text Representation I (Classic Method)
**Released:** September 28, 2026 (Week 4 – Programming)
**Deadline:** October 5, 2026 (Mon), 23:59
**Total:** 100 points

## 🎯 Objectives

Assignment 1 turned raw tweets into clean token lists. This assignment turns those tokens into the
**vector / matrix representations** of lecture 4-1:

- Build a **term-document matrix** by hand, in both the **binary** and the **frequency** form
- Measure the **sparsity** of that matrix
- Compute **df** and **idf** yourself, and then **TF-IDF**
- Compare what a document looks like when ranked by raw count and by TF-IDF
- Read scikit-learn's `CountVectorizer` / `TfidfVectorizer` and its weighting **variants**
- Add **n-gram** features and measure what they cost
- Search the resulting vector space with **cosine similarity**

Everything here is inside lecture 4-1. No classifier is trained - that comes in Week 10.

## 📝 Instructions

1. `File ▸ Save a copy in Drive`, then rename the notebook to **`Assignment_2_YourName_StudentID.ipynb`** (e.g. `Assignment_2_HongGilDong_2026123456.ipynb`).
2. Fill in every `________` blank and every `# TODO` line. **Do not** change the code outside the TODO blocks, and **do not** change variable names – the grader runs your notebook.
3. Every code cell must run without errors from top to bottom (`Runtime ▸ Restart and run all`). **Keep the outputs** in the notebook when you submit.
4. Write your answers to the discussion questions in the markdown cells at the end.
5. Submit the `.ipynb` file to **"Programming Assignment 2 – Text Representation"** under *Assignments*.

**Grading** – points are given per TODO item (see the table). Partial credit is given when the logic is correct but the output differs slightly.

| Part | Content | Points |
|---|---|---|
| 1 | Bag of words & the term-document matrix (TODO 1–3) | 25 |
| 2 | Word weighting: df, idf, TF-IDF (TODO 4–7) | 35 |
| 3 | N-grams (TODO 8–9) | 20 |
| 4 | The vector space (TODO 10) | 10 |
| 5 | Discussion questions (Q1–Q2) | 10 |

## 📂 Dataset

The **same** Twitter Entity Sentiment file as Assignment 1, so that you can reuse the pipeline you
already wrote. Section 0 repeats the Assignment-1 cleaning steps for you - **there is no TODO there**.

| column | description |
|---|---|
| `idx` | tweet id (duplicates removed, first kept) |
| `Entity` | the game / company the tweet is about |
| `Sentiment` | `Positive`, `Negative`, `Neutral` (`Irrelevant` dropped) |
| `Review` | the tweet text |

## &nbsp;0. Setup and the Assignment-1 pipeline
*This section is complete. No TODO items here.* It reproduces exactly what you built in Assignment 1:
cleaning, regex tokenization and stop-word removal. Run it once and check the printed shapes.

In [ ]:
import nltk

for r in ['punkt', 'punkt_tab', 'stopwords']:
    nltk.download(r, quiet=True)

import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from nltk.tokenize import RegexpTokenizer
from nltk.corpus import stopwords
from nltk import bigrams
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import warnings

warnings.filterwarnings('ignore')
%config InlineBackend.figure_format = 'retina'
%matplotlib inline

SENTIMENTS = ['Positive', 'Neutral', 'Negative']
COLORS = {'Positive': '#2a9d8f', 'Neutral': '#8d99ae', 'Negative': '#e76f51'}

In [ ]:
# --- the Assignment 1 pipeline, given ---------------------------------------
DATA_URL = "https://raw.githubusercontent.com/snhzyn/2026-2-Unstructured-Data-Analysis/main/week03-text-preprocessing/data/twcs.csv"

data = pd.read_csv(DATA_URL)
data_unique = data.drop_duplicates(subset='idx', keep='first')
df = data_unique[data_unique['Sentiment'] != 'Irrelevant'].reset_index(drop=True)


def clean_text(d):
    d = str(d).lower()
    d = re.sub(r"http\S+|www\S+|\S+\.com\S*", " ", d)
    d = re.sub(r"[^a-z'\s]", " ", d)
    return d


CUSTOM_STOPS = ['im', 'ive', 'dont', 'cant', 'thats', 'like', 'get', 'got', 'one',
                'game', 'games', 'amp']
stop_words = set(stopwords.words('english')) | set(CUSTOM_STOPS)

tokenizer = RegexpTokenizer(r"[a-z']{2,}")

df['clean'] = df['Review'].apply(clean_text)
df['tokens'] = df['clean'].apply(tokenizer.tokenize)
df['tokens_clean'] = df['tokens'].apply(lambda ts: [t for t in ts if t not in stop_words])

print('documents :', df.shape)
print('per class :', dict(df['Sentiment'].value_counts()))
print('tokens    :', df['tokens_clean'].apply(len).sum())
df[['Sentiment', 'Review', 'tokens_clean']].head(3)

## &nbsp;1. Bag of Words & the Term-Document Matrix (25 pts)

### TODO 1: Build the vocabulary (7 points)

Count how often every token appears in the **whole corpus** (`tokens_clean`), sort the words by
frequency in **descending** order, and keep the **top 500** as the feature list `word_features`.

Remember the lecture: the feature list has a **fixed order**, and that order defines what each
position of a document vector means.

**Expected output:** `#distinct words in the corpus: 16523` and the first five features
`['johnson', 'play', 'good', 'new', 'love']`

In [ ]:
# TODO 1: corpus word counts -> sorted feature list -> top 500

word_count = {}
for tokens in df['tokens_clean']:
    for w in tokens:
        word_count[w] = word_count.get(w, 0) ________          # hint: one more occurrence

sorted_features = sorted(word_count, key=________, reverse=True)   # hint: sort BY the count of each word
word_features = sorted_features[________]                          # hint: the first 500

print('#distinct words in the corpus:', len(sorted_features))
print('first five features:', word_features[:5])

### TODO 2: Count vector and binary vector by hand (10 points)

Write the two functions of the lecture slide:

* `frequency_vector(tokens)` – how many times each feature occurs in the document
* `binary_vector(tokens)` – `1` if the feature occurs at all, `0` otherwise

Both return a list of length 500, in the order of `word_features`.

**Expected output** for the test document `['play', 'play', 'new']`:
`frequency [0, 2, 0, 1, 0]` and `binary [0, 1, 0, 1, 0]` for the first five features
(`play` is feature 1 and `new` is feature 3, so position 0 stays empty).

In [ ]:
# TODO 2: the two representations of the lecture slide

def frequency_vector(tokens):
    counts = Counter(tokens)
    return [________ for w in word_features]        # hint: the count of w, 0 when absent (Counter[w] already does this)


def binary_vector(tokens):
    return [________ for w in word_features]        # hint: 1 if w occurs in tokens, else 0


test_doc = ['play', 'play', 'new']
print('frequency', frequency_vector(test_doc)[:5])
print('binary   ', binary_vector(test_doc)[:5])

### TODO 3: The matrix, and how empty it is (8 points)

Apply `frequency_vector` to every document to obtain `X_manual` (a list of 10,282 lists), then
report the **sparsity**: how many of all the cells are `0`.

**Expected output:** shape `(10282, 500)` and `zeros: 5092352 of 5141000 -> 99.1% of the matrix is empty`

In [ ]:
# TODO 3: build the whole matrix and measure its sparsity

X_manual = [________ for tokens in df['tokens_clean']]     # hint: the frequency vector of each document

n_rows, n_cols = len(X_manual), len(X_manual[0])
zeros = sum(row.count(0) for row in X_manual)

print('shape:', (n_rows, n_cols))
print(f'zeros: {zeros} of {________} -> {zeros / ________ :.1%} of the matrix is empty')

## &nbsp;2. Word Weighting (35 pts)

### TODO 4: Document frequency and IDF (10 points)

$\text{df}_t$ is the number of **documents** in which the term appears - not how often it appears.
Then $\text{idf}_t = \log_{10}(N / \text{df}_t)$, exactly the lecture formula.

Careful with `df`: a word occurring five times in one tweet still counts as **one** document.

**Expected output:** `play: df=484 idf=1.327` and the five lowest-idf words
`['play', 'good', 'new', 'love', 'shit']`

> 🤔 Compare this list with `word_features[:5]` from TODO 1. `johnson` is the **most frequent** word of the
> corpus but it is *not* the lowest-idf word - it is repeated many times inside a few tweets
> (*Johnson & Johnson*). tf and df are not the same thing.

In [ ]:
# TODO 4: df and idf

N = len(df)

df_t = {}
for tokens in df['tokens_clean']:
    for w in ________(tokens):                     # hint: each document counts ONCE, whatever the token repeats
        df_t[w] = df_t.get(w, 0) + 1

idf_t = {w: ________ for w, d in df_t.items()}     # hint: np.log10 of N divided by d

print(f"play: df={df_t['play']} idf={idf_t['play']:.3f}")
print('lowest idf :', sorted(word_features, key=lambda w: idf_t[w])[:5])

### TODO 5: TF-IDF of one document, by hand (10 points)

Take document `DOC_ID` and compute `tf * idf` for every token it actually contains.
Print the top 5 by **raw count** and the top 5 by **TF-IDF**, so that the two rankings can be compared.

**Expected output** for `DOC_ID = 893`:
by count `['amazon', 'great', 'shivaji', 'maharashtra', 'maratha']`,
by TF-IDF `['shivaji', 'maharashtra', 'maratha', 'great', 'amazon']`

Note where `amazon` ends up. Keep this document in mind for Question 1.

In [ ]:
# TODO 5: TF-IDF of a single document

DOC_ID = 893
tokens = df['tokens_clean'][DOC_ID]
print('tweet:', df['Review'][DOC_ID][:120])
print('tokens:', tokens)

counts = Counter(tokens)
tfidf = {w: ________ for w, c in counts.items() if w in idf_t}   # hint: the lecture formula, tf times idf

top_count = sorted(counts, key=counts.get, reverse=True)[:5]
top_tfidf = sorted(tfidf, key=________, reverse=True)[:5]        # hint: sort BY the tf-idf value

print('by count :', top_count)
print('by TF-IDF:', top_tfidf)

### TODO 6: `CountVectorizer` and `TfidfVectorizer` (8 points)

Now the scikit-learn route. Both vectorizers must use **our own tokens**, not their default tokenizer.
The usual trick: join each token list back into a string and split on white space
(`tokenizer=str.split`, `token_pattern=None`, `lowercase=False`) - the text is already clean and lowercase.

Build
* `X_count` with `CountVectorizer` (frequency representation), and
* `X_tfidf` with `TfidfVectorizer`,

both restricted to `vocabulary=word_features`.

**Expected output:** both `(10282, 500)`, `X_count` density `0.95%`, and the manual and sklearn
count vectors of document 0 identical (`True`).

In [ ]:
# TODO 6: the same two matrices with scikit-learn

df['joined'] = df['tokens_clean'].apply(' '.join)     # token list -> "play play new" (given)

cv = CountVectorizer(vocabulary=word_features, tokenizer=str.split,
                     token_pattern=None, lowercase=False)
X_count = cv.________(df['joined'])                   # hint: learn and transform in one call

tv = TfidfVectorizer(vocabulary=________, tokenizer=str.split,
                     token_pattern=None, lowercase=False)
X_tfidf = tv.fit_transform(df['joined'])

print('X_count:', X_count.shape, ' X_tfidf:', X_tfidf.shape)
print(f'X_count density: {X_count.nnz / (X_count.shape[0] * X_count.shape[1]):.2%}')
print('manual == sklearn:', (X_count.toarray()[0] == np.array(X_manual[0])).all())

### TODO 7: Which words describe each sentiment? (7 points)

Sum each column of `X_count` and of `X_tfidf` **over the rows of one sentiment class**, and report the
top 8 features of each class under both weightings. Use `X[mask].sum(axis=0)`, where `mask` is a boolean
array that is `True` for the rows of that class.

**Expected output:** for *Positive*, by count the list ends with `..., play, fun`, while by TF-IDF
`play` drops out of the top 8 altogether and `wait, thank` appear instead.

In [ ]:
# TODO 7: top-8 features per sentiment, by count and by TF-IDF

names = np.array(cv.get_feature_names_out())

for s in SENTIMENTS:
    mask = (df['Sentiment'] == s).values
    by_count = np.asarray(X_count[mask].sum(axis=0)).ravel()
    by_tfidf = np.asarray(X_tfidf[mask].________(axis=0)).ravel()     # hint: the same operation on the TF-IDF matrix

    top_c = names[np.argsort(-by_count)[:8]]
    top_t = names[np.argsort(________)[:8]]                            # hint: sort descending, like the line above

    print(f'[{s}]')
    print('   by count :', list(top_c))
    print('   by TF-IDF:', list(top_t))

## &nbsp;3. N-Grams (20 pts)

### TODO 8: What n-grams cost (10 points)

Bag of words threw the word order away. An n-gram buys a little of it back, and the price is the
number of features. Build a `TfidfVectorizer` for `ngram_range` `(1,1)`, `(1,2)` and `(1,3)` with
`min_df=2`, and report the number of features **and the density** of each matrix.

Do **not** pass `vocabulary=` here - we want the vectorizer to discover the n-grams itself.

**Expected output:**
`(1, 1) ->   6,760 features, density 0.1254%`
`(1, 2) ->  12,456 features, density 0.0841%`
`(1, 3) ->  13,728 features, density 0.0797%`

In [ ]:
# TODO 8: feature count and density for three n-gram ranges

for ngram in [(1, 1), (1, 2), (1, 3)]:
    vec = TfidfVectorizer(tokenizer=str.split, token_pattern=None, lowercase=False,
                          ngram_range=________, min_df=2)             # hint: the loop variable
    X = vec.fit_transform(df['joined'])
    density = X.nnz / (X.shape[0] * X.shape[1])
    print(f'{ngram} -> {X.shape[1]:>7,} features, density {density:.4%}')

### TODO 9: The most characteristic bigram of each sentiment (10 points)

Build the bigrams **inside each tweet** (so that the last word of one tweet is never paired with the
first word of the next), count them per sentiment class, and print the 8 most frequent bigrams of each class.

**Expected output:** *Neutral* starts with `(('johnson', 'johnson'), 161)` and *Negative* with
`(('home', 'depot'), 87)`.

In [ ]:
# TODO 9: top-8 bigrams per sentiment

for s in SENTIMENTS:
    pairs = Counter()
    for tokens in df.loc[df['Sentiment'] == s, 'tokens_clean']:
        pairs.update(________(tokens))              # hint: the bigrams of ONE tweet (imported at the top)

    print(f'[{s}]', pairs.________(8))              # hint: the 8 most frequent

## &nbsp;4. The Vector Space (10 pts)

### TODO 10: Cosine similarity search (10 points)

With TF-IDF each tweet is a point in a 500-dimensional space. Find the 5 tweets closest to tweet
`QUERY_ID` by **cosine similarity**, and print each neighbor's similarity, sentiment and text.

`TfidfVectorizer` already l2-normalizes the rows, so the cosine is simply the dot product - but use
`cosine_similarity` so that the code says what it means.

**Expected output:** with `QUERY_ID = 1525` the three closest tweets all have `cos = 0.879`.

In [ ]:
# TODO 10: the 5 nearest tweets to the query

QUERY_ID = 1525

sim = cosine_similarity(X_tfidf[________], X_tfidf)[0]     # hint: the query row of the matrix
order = np.argsort(-sim)[________]                         # hint: highest first, and skip the query itself

print('QUERY :', df['Review'][QUERY_ID][:110])
print()
for rank, i in enumerate(order, 1):
    print(f'{rank}. cos={sim[i]:.3f}  [{df["Sentiment"][i]:8}] {df["Review"][i][:90]}')

## &nbsp;5. Discussion Questions (10 pts)
Write your answers in the markdown cells below (3–5 sentences each). Base your answers on **your own outputs** above.

### Question 1 (5 points) – count vs TF-IDF

Look at your TODO 7 output for **one** sentiment class of your choice.

(a) Name one word that is high by raw count but drops when TF-IDF is used, **and quote its df or idf from TODO 4**.
(b) Explain, using the lecture formula, why TF-IDF moves that word down.
(c) `CUSTOM_STOPS` already removed words such as `game` and `like`. In one sentence: if we had used TF-IDF
from the start, would we still have needed that stop-word list? Why (not)?

*[Write your answer here]*

In [ ]:
# Question 1 - run this cell and use the numbers in your answer

for w in ['johnson', 'play', 'good', 'new', 'amazon', 'shivaji']:
    print(f"{w:6} df={df_t[w]:5}  idf={idf_t[w]:.3f}")

### Question 2 (5 points) – the price of the representation

(a) Report the density of your `(1, 1)` and `(1, 3)` matrices from TODO 8.
(b) The lecture ends the chapter with two warnings about this vector space: *"very high dimensional"*
and *"sparseness: most entries are zero"*. Explain, with your own two numbers, why adding trigrams makes
**both** warnings worse rather than better.
(c) In TODO 10 the five neighbors all carry the same `Sentiment` label. Explain what the cosine similarity
is actually measuring in that result, and say whether you would expect the same for a query tweet about a
product that people both praise and complain about.

*[Write your answer here]*

---
### ✅ Before you submit

- [ ] `Runtime ▸ Restart and run all` finishes with no error
- [ ] Every `________` is filled in and every output cell is visible
- [ ] Both discussion answers quote numbers from **your own** outputs
- [ ] The file is named `Assignment_2_YourName_StudentID.ipynb`